# 03 Features

**Purpose.** Build one feature table per split: one row per (search, candidate) pair with the label and all model inputs. Features are organised in **groups** so that `05_models` can add them step by step (the LightGBM ladder in PROJECT_PLAN section 5) and measure what each kind of information adds.

| Group | Features | Ladder step | Basis |
|---|---|---|---|
| **G0 search** | query length, list size | base | constant within a list; only useful in interactions |
| **G1 relevance** | query-title shared terms, coverage, Jaccard, BM25, title length; query term in brand / L3 category name; candidate in catalogue | base (non-personal) | FINDINGS 4.5, lecturer hint #7 |
| **G2 popularity** | exposures, smoothed engagement rate, history popularity; **training records only, out-of-fold** | base (non-personal) | FINDINGS 3 |
| **G3 pooled history** | matching history actions over all history: product, brand, shop, L1-L4 path; history length | step 2 (Team 3-like) | FINDINGS 4 |
| **G4 windows** | per window x type block (0-7 d CLICK / CART / ORD, 7-90 d CART / ORD, 90-365 d ORD, FLW): same product, same shop (other products), same brand (other products), same L3 path; days since the latest same product / shop / brand action; block sizes | step 3 | FINDINGS 7, 7.1 |
| **G5 context** | days since the last action, actions in the last 7 days, share of query-less history, type of the last action | step 4 (context test) | FINDINGS 6, E04/E06 |
| **G6 query-history** | max Jaccard between the current and past queries, any overlap; product / shop / brand matches from history actions whose query shares a term with the current one | step 5 | FINDINGS 6 |

**Rules.**
- Candidate order is shuffled within each search (seeded) before anything else; no feature uses a position.
- Missing catalogue metadata gives `NaN`, never 0 ("unknown", not "no match"). Category matches use the full ancestor path.
- Population statistics (G2) come from training records only. For training rows they are **out-of-fold** (5 folds by search, `session_id % 5`): a training search never sees its own labels or history. Counts are rescaled by 5/4 so training and validation rows are on the same scale; the smoothed engagement rate uses a prior of 20 exposures at the global training engagement rate.
- BM25 (k1 = 1.2, b = 0.75) uses document frequencies and average title length over the **whole catalogue** (12.1M titles; no labels involved).

**Outputs** (under the Git-ignored `data/processed/`): `features_v1_train_dev.parquet`, `features_v1_validation.parquet` and `features_v1_groups.json` (feature names per group). The test split is built with the same code only for the final models (`BUILD` below).

**Run time:** two passes over the behaviour file, one over the catalogue, then the features; roughly 15-20 minutes locally.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import math
import sys
import time

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
else:
    DATA_DIR = REPO_ROOT / "data"
BEHAVIOUR, CATALOGUE = DATA_DIR / "user_behavior_data.txt", DATA_DIR / "product_meta_data.txt"
PROCESSED = REPO_ROOT / "data/processed"

sys.path.insert(0, str(REPO_ROOT))
from scripts.jd_data import iter_records, SECONDS_PER_DAY
from scripts.evaluation import evaluate, summarise
from scripts.history_match_diagnostics import metadata_keys, validate_match_helpers
from scripts.query_diagnostics import overlap_features, token_set, tokenize, validate_helpers

validate_helpers()
validate_match_helpers()
SEED, N_FOLDS, PRIOR_STRENGTH = 4222, 5, 20
K1, B = 1.2, 0.75
DAY = SECONDS_PER_DAY
pd.options.display.float_format = "{:,.4f}".format


def split_terms(text):
    # Catalogue texts: plain split on the term separator (no validation, so one malformed title cannot stop the scan).
    return text.split("") if text and text != "-1" else []


assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
BUILD = {  # split name -> search IDs; add "test" only for the final models
    "train_dev": assignment.index[assignment.dev_subsample].to_numpy(),
    "validation": assignment.index[assignment.split == "validation"].to_numpy(),
}
train_ids = set(assignment.index[assignment.split == "train"].tolist())
assert all(sid in train_ids for sid in BUILD["train_dev"])
print({name: len(ids) for name, ids in BUILD.items()})

{'train_dev': 40000, 'validation': 20000}


## 1. Load the target searches

One pass over the behaviour file keeps the searches we build features for. Candidates are shuffled within each search here, once, with a fixed seed.

In [2]:
target = {int(sid): name for name, ids in BUILD.items() for sid in ids}
rng = np.random.default_rng(SEED)
records = []
for session_id, fields in iter_records(BEHAVIOUR):
    if session_id not in target:
        continue
    candidates = np.array(fields[1].split("_"))
    labels = np.array([int(float(v)) for v in fields[2].split("_")], dtype=np.int8)
    order = rng.permutation(len(candidates))
    gaps = np.array(fields[6].split("_"), dtype=np.int64)
    records.append(dict(session_id=session_id, split=target[session_id], query=fields[0],
                        candidates=candidates[order], labels=labels[order], past=fields[4].split("_"),
                        types=np.array(fields[5].split("_")), queries=fields[3].split("_"),
                        elapsed_d=np.cumsum(gaps[::-1])[::-1][1:] / DAY, last_gap_d=gaps[-1] / DAY))
candidate_wids = {w for r in records for w in r["candidates"]}
history_wids = {w for r in records for w in r["past"]}
query_vocab = set().union(*(token_set(r["query"]) for r in records))
print(f"searches {len(records):,}; distinct candidates {len(candidate_wids):,}; "
      f"distinct history products {len(history_wids):,}; query terms {len(query_vocab):,}")

searches 60,000; distinct candidates 3,637,532; distinct history products 2,574,132; query terms 26,850


## 2. Catalogue pass

One scan of the 12.1M-row catalogue collects: metadata (brand, shop, category path) for every candidate and history product; the raw title, brand name and L3 category name for candidates; and, for BM25, the document frequency of every current-query term and the average title length over all titles.

In [3]:
start = time.time()
needed = candidate_wids | history_wids
metadata, names = {}, {}
doc_freq, n_docs, total_len = Counter(), 0, 0
with CATALOGUE.open(encoding="utf-8") as handle:
    assert next(handle).split("\t", 1)[0] == "wid"
    for line in handle:
        wid, title, rest = line.split("\t", 2)
        terms = split_terms(title)
        if terms:
            n_docs += 1
            total_len += len(terms)
            doc_freq.update(query_vocab.intersection(terms))
        if wid in needed:
            fields = line.rstrip("\r\n").split("\t")
            metadata[wid] = tuple(int(fields[j]) if fields[j] not in ("", "-1") else None for j in (2, 12, 4, 6, 8, 10))
            if wid in candidate_wids:
                names[wid] = (fields[1], fields[3], fields[9])
avg_title_len = total_len / n_docs
idf = {t: math.log(1 + (n_docs - doc_freq[t] + 0.5) / (doc_freq[t] + 0.5)) for t in query_vocab}
print(f"catalogue titles {n_docs:,}, average length {avg_title_len:.1f} terms; metadata for "
      f"{len(metadata) / len(needed):.1%} of needed products; {time.time() - start:.0f} s")

catalogue titles 12,141,247, average length 27.9 terms; metadata for 94.7% of needed products; 200 s


## 3. Training-only popularity (G2)

One pass over all 133,831 training records counts, for every target candidate product: how often it was shown, how often it was engaged and how often it appears in histories, separately per fold (`session_id % 5`). Training rows later use the other four folds only; validation rows use all training records.

In [4]:
item_index = {w: i for i, w in enumerate(sorted(candidate_wids))}
exposures = np.zeros((N_FOLDS, len(item_index)), np.int32)
engaged = np.zeros((N_FOLDS, len(item_index)), np.int32)
in_history = np.zeros((N_FOLDS, len(item_index)), np.int32)
engaged_total = shown_total = 0
for session_id, fields in iter_records(BEHAVIOUR):
    if session_id not in train_ids:
        continue
    fold = session_id % N_FOLDS
    labels = [int(float(v)) for v in fields[2].split("_")]
    engaged_total += sum(l >= 1 for l in labels)
    shown_total += len(labels)
    for wid, label in zip(fields[1].split("_"), labels):
        i = item_index.get(wid)
        if i is not None:
            exposures[fold, i] += 1
            engaged[fold, i] += label >= 1
    for wid in fields[4].split("_"):
        i = item_index.get(wid)
        if i is not None:
            in_history[fold, i] += 1
PRIOR_RATE = engaged_total / shown_total
TOTALS = {name: array.sum(axis=0) for name, array in
          [("exposures", exposures), ("engaged", engaged), ("in_history", in_history)]}
print(f"training engagement rate {PRIOR_RATE:.4f}")


def popularity(wids, session_id, split):
    idx = np.array([item_index[w] for w in wids])
    if split == "validation" or split == "test":
        exp, eng, hist, scale = TOTALS["exposures"][idx], TOTALS["engaged"][idx], TOTALS["in_history"][idx], 1.0
    else:  # out-of-fold for training rows
        fold = session_id % N_FOLDS
        exp = TOTALS["exposures"][idx] - exposures[fold, idx]
        eng = TOTALS["engaged"][idx] - engaged[fold, idx]
        hist = TOTALS["in_history"][idx] - in_history[fold, idx]
        scale = N_FOLDS / (N_FOLDS - 1)
    rate = (eng + PRIOR_STRENGTH * PRIOR_RATE) / (exp + PRIOR_STRENGTH)
    return exp * scale, rate, hist * scale

training engagement rate 0.0207


## 4. Feature definitions

All features for one search are computed by `search_features`. History blocks follow the observation windows (FINDINGS 7): clicks are only observed for 7 days, carts and follows for 90, orders for 365. A handful of clicks older than 7 days (107 in the full data) belong to no block but stay in the pooled counts.

In [5]:
LEVEL_POS = {"product": 0, "brand": 1, "shop": 2, "L1": 3, "L2": 4, "L3": 5, "L4": 6}
BLOCKS = {
    "click_0_7": lambda t, d: (t == "CLICK") & (d < 7),
    "cart_0_7": lambda t, d: (t == "CART") & (d < 7),
    "ord_0_7": lambda t, d: (t == "ORD") & (d < 7),
    "cart_7_90": lambda t, d: (t == "CART") & (d >= 7),
    "ord_7_90": lambda t, d: (t == "ORD") & (d >= 7) & (d < 90),
    "ord_90_365": lambda t, d: (t == "ORD") & (d >= 90),
    "flw": lambda t, d: t == "FLW",
}
LAST_TYPE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}


def counts(cand_keys, hist_keys, idx, pos):
    # Matching history actions (restricted to idx) per candidate at one level; NaN if the candidate key is unknown.
    counter = Counter(hist_keys[i][pos] for i in idx)
    return np.array([counter.get(k[pos], 0) if k[pos] is not None else np.nan for k in cand_keys], dtype=float)


def latest(cand_keys, hist_keys, elapsed_d, pos):
    # Days since the most recent history action matching the candidate at one level (NaN if none or unknown).
    newest = {}
    for keys, age in zip(hist_keys, elapsed_d):
        key = keys[pos]
        if key is not None and (key not in newest or age < newest[key]):
            newest[key] = age
    return np.array([newest.get(k[pos], np.nan) if k[pos] is not None else np.nan for k in cand_keys])


def relevance(query_terms, wid):
    if wid not in names:
        return (np.nan,) * 7
    title, brand, category = names[wid]
    terms = split_terms(title)
    shared, coverage, jaccard = overlap_features(query_terms, frozenset(terms))
    tf, length = Counter(terms), len(terms)
    bm25 = sum(idf[t] * tf[t] * (K1 + 1) / (tf[t] + K1 * (1 - B + B * length / avg_title_len))
               for t in query_terms if tf[t])
    return (shared, coverage, jaccard, bm25, length,
            float(bool(query_terms.intersection(split_terms(brand)))), float(bool(query_terms.intersection(split_terms(category)))))


def search_features(r):
    cands, n = r["candidates"], len(r["candidates"])
    types, elapsed_d = r["types"], r["elapsed_d"]
    query_terms = token_set(r["query"])
    cand_keys = [metadata_keys(w, metadata) for w in cands]
    hist_keys = [metadata_keys(w, metadata) for w in r["past"]]
    all_idx = np.arange(len(types))
    f = {"session_id": np.full(n, r["session_id"]), "wid": cands, "label": r["labels"]}

    # G0 search
    f["query_len"] = np.full(n, len(tokenize(r["query"])))
    f["n_candidates"] = np.full(n, n)
    # G1 relevance
    rel = np.array([relevance(query_terms, w) for w in cands], dtype=float)
    for j, name in enumerate(["title_shared", "title_coverage", "title_jaccard", "bm25", "title_len",
                              "query_in_brand", "query_in_cate3"]):
        f[name] = rel[:, j]
    f["in_catalogue"] = np.array([w in metadata for w in cands], dtype=float)
    # G2 popularity
    f["pop_exposures"], f["pop_engaged_rate"], f["pop_history"] = popularity(cands, r["session_id"], r["split"])
    # G3 pooled history
    f["hist_len"] = np.full(n, len(types))
    for level, pos in LEVEL_POS.items():
        f[f"pooled_{level}"] = counts(cand_keys, hist_keys, all_idx, pos)
    # G4 windows
    for block, rule in BLOCKS.items():
        idx = np.flatnonzero(rule(types, elapsed_d))
        product = counts(cand_keys, hist_keys, idx, 0)
        f[f"{block}_product"] = product
        f[f"{block}_shop_other"] = counts(cand_keys, hist_keys, idx, 2) - product
        f[f"{block}_brand_other"] = counts(cand_keys, hist_keys, idx, 1) - product
        f[f"{block}_L3"] = counts(cand_keys, hist_keys, idx, 5)
        f[f"n_{block}"] = np.full(n, len(idx))
    for level in ("product", "shop", "brand"):
        f[f"days_since_{level}"] = latest(cand_keys, hist_keys, elapsed_d, LEVEL_POS[level])
    # G5 context
    f["days_since_last_action"] = np.full(n, r["last_gap_d"])
    f["actions_last_7d"] = np.full(n, int((elapsed_d < 7).sum()))
    f["share_queryless"] = np.full(n, np.mean([q == "-1" for q in r["queries"]]))
    f["last_action_type"] = np.full(n, LAST_TYPE[types[-1]])
    # G6 query-history
    past_terms = [token_set(q) for q in r["queries"]]
    jaccards = [len(query_terms & p) / len(query_terms | p) for p in past_terms if p]
    f["hist_query_max_jaccard"] = np.full(n, max(jaccards, default=0.0))
    related = np.flatnonzero([bool(query_terms & p) for p in past_terms])
    f["hist_query_any_overlap"] = np.full(n, float(len(related) > 0))
    for level in ("product", "shop", "brand"):
        f[f"qrel_{level}"] = counts(cand_keys, hist_keys, related, LEVEL_POS[level])
    return pd.DataFrame(f)


GROUPS = {
    "G0_search": ["query_len", "n_candidates"],
    "G1_relevance": ["title_shared", "title_coverage", "title_jaccard", "bm25", "title_len", "query_in_brand",
                     "query_in_cate3", "in_catalogue"],
    "G2_popularity": ["pop_exposures", "pop_engaged_rate", "pop_history"],
    "G3_pooled_history": ["hist_len"] + [f"pooled_{level}" for level in LEVEL_POS],
    "G4_windows": [f"{b}_{x}" for b in BLOCKS for x in ("product", "shop_other", "brand_other", "L3")]
                  + [f"n_{b}" for b in BLOCKS] + [f"days_since_{l}" for l in ("product", "shop", "brand")],
    "G5_context": ["days_since_last_action", "actions_last_7d", "share_queryless", "last_action_type"],
    "G6_query_history": ["hist_query_max_jaccard", "hist_query_any_overlap", "qrel_product", "qrel_shop", "qrel_brand"],
}
ALL_FEATURES = [c for cols in GROUPS.values() for c in cols]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES))
print({g: len(c) for g, c in GROUPS.items()}, "total", len(ALL_FEATURES))

{'G0_search': 2, 'G1_relevance': 8, 'G2_popularity': 3, 'G3_pooled_history': 8, 'G4_windows': 38, 'G5_context': 4, 'G6_query_history': 5} total 68


Toy check of the block rules and the "other products" logic: a candidate the shopper clicked yesterday and whose shop they ordered from 100 days ago.

In [6]:
_meta = {"a": (10, 20, 1, 2, 3, 4), "b": (11, 20, 1, 2, 3, 5)}
_keys_c = [metadata_keys("a", _meta)]
_keys_h = [metadata_keys("a", _meta), metadata_keys("b", _meta)]
_types, _days = np.array(["CLICK", "ORD"]), np.array([1.0, 100.0])
assert BLOCKS["click_0_7"](_types, _days).tolist() == [True, False]
assert BLOCKS["ord_90_365"](_types, _days).tolist() == [False, True]
_idx = np.flatnonzero(BLOCKS["ord_90_365"](_types, _days))
assert counts(_keys_c, _keys_h, _idx, 2)[0] - counts(_keys_c, _keys_h, _idx, 0)[0] == 1  # shop via other product
assert latest(_keys_c, _keys_h, _days, 0)[0] == 1.0 and latest(_keys_c, _keys_h, _days, 2)[0] == 1.0
print("feature helper checks passed")

feature helper checks passed


## 5. Build and save

In [7]:
start = time.time()
parts = defaultdict(list)
for r in records:
    parts[r["split"]].append(search_features(r))
tables = {}
for split, frames in parts.items():
    table = pd.concat(frames, ignore_index=True)
    table[ALL_FEATURES] = table[ALL_FEATURES].astype(np.float32)
    table.to_parquet(PROCESSED / f"features_v1_{split}.parquet", index=False)
    tables[split] = table
    print(f"{split}: {table.session_id.nunique():,} searches, {len(table):,} rows, {len(ALL_FEATURES)} features")
(PROCESSED / "features_v1_groups.json").write_text(json.dumps(GROUPS, indent=2))
print(f"{time.time() - start:.0f} s")

train_dev: 40,000 searches, 3,573,362 rows, 68 features


validation: 20,000 searches, 1,781,573 rows, 68 features
961 s


## 6. Checks

1. **Completeness:** every search has its full candidate list (same sizes as the search table).
2. **Consistency:** pooled product matches equal the sum over blocks plus the few old clicks; "other product" counts are never negative.
3. **Train vs validation profile:** feature means and missing shares should be similar; large gaps in G2 would mean the out-of-fold scheme puts training rows on a different scale than validation rows.
4. **Sanity against E03 (training rows only):** single-feature rankings should reproduce the window re-check (e.g. pooled same-product gain about 0.09).

In [8]:
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
for split, table in tables.items():
    sizes = table.groupby("session_id").size()
    assert (sizes == searches.loc[sizes.index, "n_candidates"]).all() and len(sizes) == len(BUILD[split])
    block_sum = table[[f"{b}_product" for b in BLOCKS]].sum(axis=1)
    assert (block_sum <= table.pooled_product).all()
    assert (table.pooled_product - block_sum).sum() < 0.001 * table.pooled_product.sum()
    other = [c for c in ALL_FEATURES if c.endswith("_other")]
    assert (table[other].fillna(0) >= 0).all().all()
print("completeness and consistency checks passed")

profile = pd.DataFrame({
    (split, stat): (table[ALL_FEATURES].mean() if stat == "mean" else table[ALL_FEATURES].isna().mean())
    for split, table in tables.items() for stat in ("mean", "missing")})
profile["mean_ratio_val_to_train"] = profile[("validation", "mean")] / profile[("train_dev", "mean")]
profile.to_csv(PROCESSED / "features_v1_profile.csv")
flagged = profile[(profile.mean_ratio_val_to_train - 1).abs() > 0.1]
print(f"features whose validation mean differs from training by more than 10%: {len(flagged)}")
profile.loc[GROUPS["G2_popularity"] + flagged.index.difference(GROUPS["G2_popularity"]).tolist()]

completeness and consistency checks passed


features whose validation mean differs from training by more than 10%: 6


train_dev         validation          \
                          mean missing       mean missing   
pop_exposures           7.0988  0.0000     7.0424  0.0000   
pop_engaged_rate        0.0231  0.0000     0.0233  0.0000   
pop_history            26.1267  0.0000    25.9367  0.0000   
flw_L3                  0.0014  0.0045     0.0062  0.0044   
flw_brand_other         0.0007  0.0045     0.0009  0.0044   
flw_product             0.0000  0.0000     0.0000  0.0000   
flw_shop_other          0.0002  0.0140     0.0001  0.0141   
ord_7_90_L3             0.1064  0.0045     0.1195  0.0044   
ord_7_90_brand_other    0.0477  0.0045     0.0529  0.0044   

                     mean_ratio_val_to_train  
                                              
pop_exposures                         0.9921  
pop_engaged_rate                      1.0084  
pop_history                           0.9927  
flw_L3                                4.4400  
flw_brand_other                       1.2413  
flw_product                           1.3372  
flw_shop_other                        0.6900  
ord_7_90_L3                           1.1232  
ord_7_90_brand_other                  1.1109

In [9]:
train = tables["train_dev"]
train_sizes = searches.loc[BUILD["train_dev"], "n_candidates"]
train["score_random"] = 0.0
base = evaluate(train, "score_random", expected_sizes=train_sizes)
rows = {}
for feature in ["pooled_product", "pooled_shop", "click_0_7_product", "click_0_7_shop_other", "bm25",
                "title_jaccard", "pop_engaged_rate", "pop_exposures", "qrel_product"]:
    train["score"] = train[feature].fillna(-1)
    per = evaluate(train, "score", expected_sizes=train_sizes)
    rows[feature] = {"ndcg_at_10": per.ndcg.mean(), "gain_over_random": (per.ndcg - base.ndcg).mean()}
sanity = pd.DataFrame(rows).T
sanity

,ndcg_at_10,gain_over_random
pooled_product,0.1777,0.0944
pooled_shop,0.1708,0.0874
click_0_7_product,0.1521,0.0688
click_0_7_shop_other,0.1155,0.0321
bm25,0.0913,0.0080
title_jaccard,0.0974,0.0140
pop_engaged_rate,0.2516,0.1683
pop_exposures,0.3106,0.2273
qrel_product,0.1313,0.0480


## Summary (E07)

- **Feature set v1:** 68 features in 7 groups for 40,000 training searches (3,573,362 rows) and 20,000 validation searches (1,781,573 rows); saved as `data/processed/features_v1_*.parquet` with the groups in `features_v1_groups.json`.
- **Checks passed:** complete lists; pooled product matches equal the block sums plus the few old clicks; "other product" counts never negative; training-row single-feature gains reproduce E03 (pooled product 0.094 vs 0.093, pooled shop 0.087 vs 0.088, recent clicks 0.069 vs 0.068).
- **Out-of-fold popularity is on the same scale** for training and validation rows (means within 1%). Six window features differ by more than 10% between splits; all are rare or heavy-tailed history counts (follows; 7-90 day orders at brand / L3 level), not population statistics.
- **Popularity is the strongest single signal on training rows:** ranking by training exposures alone reaches NDCG@10 0.311 (+0.227 over random), the smoothed engagement rate 0.252 (+0.168), against +0.094 for the best personal-history signal and +0.014 for query-title Jaccard. Likely reason (hypothesis, not tested): products JD ranks highly for common queries appear in many lists and get engaged more, so exposure counts partly recover JD's own ranking and position bias, which the file does not reveal. Consequence: the non-personalised reference must include popularity, and personalisation is judged by what it adds on top.
